In [1]:
import pandas as pd
from datetime import datetime

In [2]:
# Load Data
client_df = pd.read_csv('client_data.csv') 
price_df = pd.read_csv('price_data.csv')

In [3]:
# Correct Column Names
cols_correct = {
    'price_off_peak_var': 'price_peak_var',
    'price_peak_var': 'price_mid_peak_var',
    'price_mid_peak_var': 'price_off_peak_var', 
    'price_off_peak_fix': 'price_peak_fix',
    'price_peak_fix': 'price_mid_peak_fix',
    'price_mid_peak_fix': 'price_off_peak_fix'}

price_df = price_df.rename(columns=cols_correct)
price_df.sort_values(by='price_off_peak_var')

,id,price_date,price_peak_var,price_mid_peak_var,price_off_peak_var,price_peak_fix,price_mid_peak_fix,price_off_peak_fix
0,038af19179925da21a25619c5a24b745,2025-01-01,0.151367,0.000000,0.000000,44.266931,0.000000,0.000000
111035,7590deb8871805e433f52e8acfc0225e,2025-06-01,0.153048,0.000000,0.000000,44.444710,0.000000,0.000000
111034,7590deb8871805e433f52e8acfc0225e,2025-05-01,0.153048,0.000000,0.000000,44.444710,0.000000,0.000000
111033,7590deb8871805e433f52e8acfc0225e,2025-04-01,0.153048,0.000000,0.000000,44.444710,0.000000,0.000000
111032,7590deb8871805e433f52e8acfc0225e,2025-03-01,0.153739,0.000000,0.000000,44.444710,0.000000,0.000000
...,...,...,...,...,...,...,...,...
473,33bb3af90650ac2e9ecac6ff2c975a6b,2025-06-01,0.160327,0.137606,0.104202,41.063969,24.837586,16.724391
474,33bb3af90650ac2e9ecac6ff2c975a6b,2025-07-01,0.160327,0.137606,0.104202,41.063969,24.837586,16.724391
469,33bb3af90650ac2e9ecac6ff2c975a6b,2025-02-01,0.207221,0.168256,0.114102,41.063970,24.837581,16.724389
468,33bb3af90650ac2e9ecac6ff2c975a6b,2025-01-01,0.207221,0.168256,0.114102,41.063970,24.837581,16.724389


In [4]:
# Processing Function
def feature_engineering (X):
    X['cons_last_ratio']=X['cons_last_month']/(X['cons_12m']/12)
    X['forecast_price_energy_diff']=X['forecast_price_energy_peak']-X['forecast_price_energy_off_peak']
    return X

In [5]:
# Processing Function
def add_price_features(X, price_df):
    types = ['var', 'fix']
    agg_dict = {}

    for t in types:
        off_col = f'price_off_peak_{t}'
        mid_col = f'price_mid_peak_{t}'
        peak_col = f'price_peak_{t}'
        
        mid_diff_key = f'mid_off_{t}_diff'
        peak_diff_key = f'peak_off_{t}_diff'
        
        price_df[mid_diff_key] = np.where(price_df[mid_col] > 0, 
                                            price_df[mid_col] - price_df[off_col], 0)
        
        price_df[peak_diff_key] = np.where(price_df[peak_col] > 0, 
                                             price_df[peak_col] - price_df[off_col], 0)

        agg_dict[mid_diff_key] = ['mean', 'max']
        agg_dict[peak_diff_key] = ['mean', 'max']

    grouped_prices = price_df.groupby('id').agg(agg_dict)

    grouped_prices.columns = [f"{stat}_{col}" for col, stat in grouped_prices.columns]
    grouped_prices = grouped_prices.reset_index()

    X = pd.merge(X, grouped_prices, on='id', how='left')

    numeric_cols = X.select_dtypes(include=['number']).columns
    X[numeric_cols] = X[numeric_cols].fillna(0)

    return X

In [6]:
# Processing Function
def transform_categorical_features(X):
    channel_mapping = {
        'foosdfpfkusacimwkcsosbicdxkicaua': 'Direct Sales',
        'usilxuppasemubllopkaafesmlibmsdf': 'International Broker',
        'lmkebamcaaclubfxadlmueccxoimlema': 'Local Broker',
        'ewpakwlliwisiwduibdlfmalxowmwpci': 'Government Tender',
        'epumfxlbckeskwekxbiuasklxalciiuu': 'Retailers',
        'fixdbufsefwooaasfcxdxadsiekoceaa': 'Noninstitutional Broker',
        'sddiedcslfslkckwlfkdpoeeailfpeds': 'Online',
        'MISSING': 'Untracked Channel'
    }
    X['channel_sales'] = X['channel_sales'].replace(channel_mapping)
    
    origin_mapping = {
        'kamkkxfxxuwbdslkwifmmcsiusiuosws': 'Special Offers',
        'lxidpiddsbxsbosboudacockeimpuepw': 'Seasonal Offers',
        'ldkssxwpmemidmecebumciepifcamkci': 'Referral',
        'MISSING': 'Untracked Origin',
        'usapbepcfoloekilkwsdiboslwaxobdp': 'Long-Term Agreements',
        'ewxeelcelemmiwuafmddpobolfuxioce': 'Organics'
    }
    X['origin_up'] = X['origin_up'].replace(origin_mapping)
    
    return X

In [7]:
# Processing Function
def transform_date_features(X):
    date_columns = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
    ref_date = datetime(2026, 1, 1)
    
    for col in date_columns:
        dates = pd.to_datetime(X[col], format='%Y-%m-%d')
        
        year_diff = ref_date.year - dates.dt.year
        month_diff = ref_date.month - dates.dt.month
        months = year_diff * 12 + month_diff
        months -= (ref_date.day < dates.dt.day).astype(int)
        
        if col == 'date_activ':
            X['months_activ'] = months
        elif col == 'date_end':
            X['months_to_end'] = -months
        elif col == 'date_modif_prod':
            X['months_modif_prod'] = months
        elif col == 'date_renewal':
            X['months_renewal'] = months
            
    return X

import numpy as np

In [8]:
# Data Transformation
EDA_df = feature_engineering(client_df)
EDA_df = add_price_features(EDA_df, price_df)
EDA_df = transform_categorical_features(EDA_df)
EDA_df = transform_date_features(EDA_df)

In [9]:
import sweetviz as sv
report = sv.analyze(EDA_df)
report.show_html('EDA.html')

                                             |          | [  0%]   00:00 -> (? left)

Report EDA.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [10]:
# Save Dataset
EDA_df.to_csv('EDA_dataset.csv', index=False)